In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# 1. Random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# 2. Load official UK road safety 2022 dataset
ALT_RAW_PATH = "https://data.dft.gov.uk/road-accidents-safety-data/dft-road-casualty-statistics-collision-2022.csv"
print("Loading official UK road safety 2022 data...")
alt_raw_df = pd.read_csv(ALT_RAW_PATH, low_memory=False)
print("Data loaded successfully! Shape:", alt_raw_df.shape)

# Map columns to match project naming
raw_df = alt_raw_df.rename(columns={
    'collision_index': 'accident_index',
    'collision_severity': 'accident_severity'
})

In [ ]:
# 3. Basic preprocessing
raw_df['hour'] = pd.to_datetime(raw_df['time'], format='%H:%M', errors='coerce').dt.hour
raw_df['month'] = pd.to_datetime(raw_df['date'], format='%d/%m/%Y', errors='coerce').dt.month

# Map severity values: 1 = Fatal, 2 = Serious, 3 = Slight
sev_map = {1: 'Fatal', 2: 'Serious', 3: 'Slight'}
raw_df['accident_severity'] = raw_df['accident_severity'].map(sev_map)

contract_cols = [
    'accident_index', 'latitude', 'longitude', 'hour', 'day_of_week', 'month',
    'speed_limit', 'road_type', 'light_conditions', 'weather_conditions',
    'road_surface_conditions', 'urban_or_rural_area', 'number_of_vehicles',
    'number_of_casualties', 'accident_severity'
]

df = raw_df[contract_cols].rename(columns={'day_of_week': 'weekday'})
df = df.replace(-1, np.nan).dropna()
print("Rows after dropping missing values:", len(df))

In [ ]:
# 4. Create binary target (1 = Severe/Fatal, 0 = Slight) and prevent data leakage
df['target_severe'] = df['accident_severity'].apply(
    lambda x: 1 if x in ['Fatal', 'Serious'] else 0
)

print("\nClass Distribution (%):")
print((df['target_severe'].value_counts(normalize=True) * 100).round(2))

# Exclude number_of_casualties to prevent data leakage
drop_cols = ['accident_index', 'accident_severity', 'number_of_casualties', 'target_severe']
X_raw = df.drop(columns=drop_cols)
y = df['target_severe']

In [ ]:
# 5. One-Hot Encoding for categorical variables
categorical_cols = [
    'weekday', 'road_type', 'light_conditions',
    'weather_conditions', 'road_surface_conditions', 'urban_or_rural_area'
]
for col in categorical_cols:
    if col in X_raw.columns:
        X_raw[col] = X_raw[col].astype(str)

X = pd.get_dummies(X_raw, columns=categorical_cols, drop_first=True)
print("Final features shape:", X.shape)

In [ ]:
# 6. Train-test split (80% train / 20% test with stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

In [ ]:
# 7. Hyperparameter tuning for max_depth using a Validation set (no test leakage)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.25,
    stratify=y_train,
    random_state=RANDOM_STATE
)

depths = [2, 3, 4, 5, 6, 8, 10]
results = []

for d in depths:
    clf = DecisionTreeClassifier(
        max_depth=d,
        class_weight='balanced',
        random_state=RANDOM_STATE
    )
    clf.fit(X_tr, y_tr)
    val_pred = clf.predict(X_val)

    results.append({
        'max_depth': d,
        'Train_F1': f1_score(y_tr, clf.predict(X_tr)),
        'Val_F1': f1_score(y_val, val_pred),
        'Val_Recall': recall_score(y_val, val_pred),
        'Val_Precision': precision_score(y_val, val_pred)
    })

tuning_df = pd.DataFrame(results)
print("\nTuning Results (Validation Set):")
print(tuning_df.round(4).to_string(index=False))

best_depth = int(tuning_df.loc[tuning_df['Val_F1'].idxmax(), 'max_depth'])
print(f"\nBest Depth Selected: {best_depth}")

# Plot Train vs Validation F1 to check overfitting
plt.figure(figsize=(7, 4))
plt.plot(tuning_df['max_depth'], tuning_df['Train_F1'], marker='o', label='Train F1')
plt.plot(tuning_df['max_depth'], tuning_df['Val_F1'], marker='o', label='Validation F1')
plt.xlabel('max_depth')
plt.ylabel('F1 score')
plt.title('F1 Score by Tree Depth')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 8. Train final model on full X_train and evaluate ONCE on X_test
final_dt = DecisionTreeClassifier(
    max_depth=best_depth,
    class_weight='balanced',
    random_state=RANDOM_STATE
)
final_dt.fit(X_train, y_train)
y_pred = final_dt.predict(X_test)

baseline_acc = 1 - y_test.mean()
print("\n=== Final Model Evaluation (Test Set) ===")
print(f"Baseline Accuracy (Always Slight): {baseline_acc:.4f}")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred, target_names=['Slight (0)', 'Severe (1)']))

In [ ]:
# 9. Plotting Results (Confusion Matrix & Top 10 Features)
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted Slight', 'Predicted Severe'],
    yticklabels=['Actual Slight', 'Actual Severe']
)
plt.title(f'Confusion Matrix (max_depth={best_depth})')
plt.ylabel('Actual')
plt.xlabel('Predicted')

plt.subplot(1, 2, 2)
importances = pd.Series(final_dt.feature_importances_, index=X.columns)
top_10 = importances.sort_values(ascending=False).head(10)
top_10.sort_values().plot(kind='barh', color='teal')
plt.title('Top 10 Most Important Features')
plt.xlabel('Importance')

plt.tight_layout()
plt.show()

In [ ]:
# 10. Example: Predict severity for a single new accident sample
sample_accident = X_test.iloc[[0]]
pred_class = final_dt.predict(sample_accident)[0]
pred_prob = final_dt.predict_proba(sample_accident)[0][1]
print(f"\nSample Prediction -> {'Severe (1)' if pred_class == 1 else 'Slight (0)'} (Probability of Severe: {pred_prob:.2%})")